# GrumpReLU Implementation

JumpReLU is a really useful activation function, but it is tailored for single dimensional features. This notebook implements and tests a variant of JumpReLU for *groups* of latent variables.

In [32]:
import torch
import numpy as np
from typing import Any

The first thing we need to implement is an approximation for the Dirac delta function using the Rectangle function. JumpReLU uses the Heaviside step function, who's derivative is the Dirac delta, which is entirely unworkable for SGD methods. 

In [ ]:
def rectangle_bandwidth(x : torch.Tensor, bandwidth : float) -> torch.Tensor:
    rectangle = (-bandwidth/2 < x) & (x < bandwidth / 2)
    
    return rectangle / bandwidth

In [ ]:
def grump_relu_forward(x : torch.Tensor, threshold : torch.Tensor):
    """
    GrumpReLU forward implementation. 
    
    Defined as a separate function since it will be used both in the forward pass of the Autograd function as well as in the GrumpReLU layer.
    """

    norms = x.norm(dim=-1)
    mask = norms > threshold
    
    return x * mask.unsqueeze(-1) 

class GrumpReLU(torch.autograd.Function):
    @staticmethod
    def forward(
        x : torch.Tensor, 
        threshold : torch.Tensor, 
        bandwidth : float
        ) -> torch.Tensor:
        return grump_relu_forward(x, threshold)

    @staticmethod
    def setup_context(
        ctx : Any, 
        inputs: tuple[torch.Tensor, torch.Tensor, float], 
        output: torch.Tensor
        ) -> None:
        x, threshold, bandwidth = inputs
        del output

        ctx.save_for_backward(x, threshold)
        ctx.bandwidth = bandwidth

    @staticmethod
    def backward(
        ctx : Any, 
        grad_outputs : torch.Tensor
        ) -> tuple[torch.Tensor, torch.Tensor, None]:

        x, threshold = ctx.saved_tensors
        bandwidth = ctx.bandwidth

        # x : ..., n_experts, d_expert
        # threshold: n_experts

        norms = x.norm(dim=-1) # n_experts

        # TODO: Finnish grump relu implementation
        x_grad = (norms > threshold).to(x).unsqueeze(-1) * grad_outputs
    
        threshold_grad 

        return (x_grad, threshold_grad, None)


In [61]:
x = torch.randn((100, 3))

threshold =  (10 - 0.5) * torch.randn((100)) + 0.5

x = grump_relu_forward(x, threshold, 0.0)

torch.count_nonzero(x)

tensor(180)